# 07 - Preprocessing Komentar (untuk IndoBERTweet)

Tahap 3.1.3 (Pra Pemrosesan Data) pada proposal TA. Membersihkan komentar hasil scraping, lalu
menyiapkan sample gold-standard untuk labeling manual (tahap 08).

**Keputusan pembersihan (sesuai catatan asli):**
- Emoji DIBIARKAN (tidak dihapus).
- URL dan mention (`@username`) DIHAPUS.
- Tanpa stemming / normalisasi kata gaul (IndoBERT pakai contextual embedding).
- Filter bahasa Indonesia opsional, default OFF (`USE_LANG_FILTER`).

**Alur:**
1. `run_eda()` -- ringkasan video target vs video yang benar-benar punya komentar, per influencer.
2. `clean_comments()` -- 6 langkah pembersihan berurutan (kosong -> URL/mention -> duplikat ->
   tanpa huruf -> <2 kata -> bahasa) -> `komentar_bersih.csv`.
3. `prepare_labeling_sample()` -- hitung ukuran sampel minimum (margin of error 3%, confidence
   95%), ambil sample acak (gold standard) untuk dilabeli manual oleh 2 annotator di tahap 08.

**Konsolidasi dari:** `02_IndoBert/01_praprocessing/praprocessing_komentar.ipynb`

**Catatan eksekusi:** semua fungsi skip otomatis kalau output sudah ada.

In [ ]:
import math
import re
from pathlib import Path

import pandas as pd

# -------------------------------------------------------------------------
# KONFIGURASI PATH
# -------------------------------------------------------------------------
FINAL_OUTPUT_ROOT = Path.cwd() / "output"
OUTPUT_DIR = FINAL_OUTPUT_ROOT / "07_preprocessing_komentar"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMMENTS_CSV = FINAL_OUTPUT_ROOT / "06_scraping_komentar" / "comments.csv"
TO_SCRAPE_CSV = FINAL_OUTPUT_ROOT / "06_scraping_komentar" / "to_scrape.csv"

EDA_PER_INFLUENCER = OUTPUT_DIR / "eda_per_influencer.csv"
LOG_PRAPROCESSING = OUTPUT_DIR / "log_praprocessing.csv"
RINGKASAN_PRAPROCESSING = OUTPUT_DIR / "ringkasan_praprocessing.csv"
KOMENTAR_PER_INFLUENCER = OUTPUT_DIR / "komentar_per_influencer.csv"
KOMENTAR_BERSIH = OUTPUT_DIR / "komentar_bersih.csv"
SAMPEL_LABELING = OUTPUT_DIR / "sampel_labeling.csv"

MIN_WORDS = 2
USE_LANG_FILTER = False

# Parameter ukuran sampel (margin of error 3%, confidence 95%)
Z_SCORE = 1.96
P_PROPORTION = 0.5
MARGIN_OF_ERROR = 0.03
N_SAMPLE_PAKAI = 2000  # dipakai asalkan >= sampel minimum
UJI_ULANG_PROP = 0.10  # 10% sampel dilabeli dua kali untuk cek konsistensi
SAMPLE_SEED = 42

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
MENTION_RE = re.compile(r"@[A-Za-z0-9_.]+")
SPACE_RE = re.compile(r"\s+")
print("Config siap.")

## STEP 1 - EDA: target scraping vs komentar yang benar-benar ada

In [ ]:
def run_eda(output_csv: Path = EDA_PER_INFLUENCER) -> pd.DataFrame:
    if output_csv.exists():
        print(f"[EDA] Sudah ada -> {output_csv} (skip)")
        return pd.read_csv(output_csv)

    comments = pd.read_csv(COMMENTS_CSV, dtype=str, keep_default_na=False, encoding="utf-8")
    to_scrape = pd.read_csv(TO_SCRAPE_CSV, dtype=str, keep_default_na=False, encoding="utf-8")

    target = to_scrape.groupby("username").size().rename("target_video")
    per_video = comments.groupby(["username", "video_id"]).size().rename("n_komentar").reset_index()
    ada_komentar = per_video.groupby("username").agg(
        video_ada_komentar=("video_id", "nunique"),
        total_komentar=("n_komentar", "sum"),
        median_komentar_per_video=("n_komentar", "median"),
    )

    eda = (pd.concat([target, ada_komentar], axis=1).fillna(0).reset_index()
           .rename(columns={"index": "username"}).sort_values("total_komentar"))
    eda["video_hilang"] = eda["target_video"] - eda["video_ada_komentar"]

    eda.to_csv(output_csv, index=False, encoding="utf-8")
    print(f"[EDA] {len(eda)} influencer | video target: {int(eda.target_video.sum()):,} | "
          f"video dengan komentar: {int(eda.video_ada_komentar.sum()):,} -> {output_csv}")
    return eda


eda = run_eda()

## STEP 2 - Pembersihan komentar (6 langkah berurutan)

In [ ]:
def clean_comments(output_csv: Path = KOMENTAR_BERSIH) -> pd.DataFrame:
    if output_csv.exists():
        print(f"[Cleaning] Sudah ada -> {output_csv} (skip)")
        return pd.read_csv(output_csv)

    comments = pd.read_csv(COMMENTS_CSV, dtype=str, keep_default_na=False, encoding="utf-8")
    df = comments.copy()
    df["comment_text_raw"] = df["comment_text"]
    awal = len(df)
    log = []

    def catat(langkah, sebelum, sesudah):
        log.append({"langkah": langkah, "sebelum": sebelum, "sesudah": sesudah,
                     "dibuang": sebelum - sesudah, "total_awal": awal,
                     "total_dibersihkan": awal - sesudah, "total_akhir": sesudah})
        print(f"[Cleaning] {langkah} ({sebelum:,} -> {sesudah:,}, dibuang {sebelum - sesudah:,})")

    catat("0. Data mentah", len(df), len(df))

    n0 = len(df)
    df = df[df["comment_text"].str.strip() != ""]
    catat("1. Komentar kosong", n0, len(df))

    n0 = len(df)
    df = df.assign(comment_text=(
        df["comment_text"].str.replace(URL_RE, " ", regex=True)
        .str.replace(MENTION_RE, " ", regex=True)
        .str.replace(SPACE_RE, " ", regex=True).str.strip()
    ))
    df = df[df["comment_text"] != ""]
    catat("2. Hapus URL/mention, jadi kosong", n0, len(df))

    n0 = len(df)
    df = df.drop_duplicates(subset=["video_id", "commenter_username", "comment_text"], keep="first")
    catat("3. Duplikat (video+akun+teks sama)", n0, len(df))

    n0 = len(df)
    df = df[df["comment_text"].str.contains(r"[A-Za-z]", regex=True)]
    catat("4. Tanpa huruf (emoji/tanda baca saja)", n0, len(df))

    n0 = len(df)
    df = df[df["comment_text"].str.split().str.len() >= MIN_WORDS]
    catat(f"5. Kurang dari {MIN_WORDS} kata", n0, len(df))

    if USE_LANG_FILTER:
        from langdetect import DetectorFactory, detect
        DetectorFactory.seed = 42

        def is_indonesia(teks):
            try:
                return detect(teks) in ("id", "ms")
            except Exception:
                return False

        n0 = len(df)
        df = df[df["comment_text"].map(is_indonesia)]
        catat("6. Bukan bahasa Indonesia", n0, len(df))
    else:
        print("[Cleaning] 6. Filter bahasa dilewati (USE_LANG_FILTER=False)")

    komentar_bersih = df[["username", "video_id", "commenter_username",
                           "comment_text_raw", "comment_text"]].rename(
        columns={"comment_text": "comment_text_clean"})
    komentar_bersih.to_csv(output_csv, index=False, encoding="utf-8")
    pd.DataFrame(log).to_csv(LOG_PRAPROCESSING, index=False, encoding="utf-8")

    eda = run_eda()
    akhir = (komentar_bersih.groupby("username")
             .agg(video_setelah_bersih=("video_id", "nunique"),
                  komentar_setelah_bersih=("comment_text_clean", "size")).reset_index())
    ringkasan = eda.merge(akhir, on="username", how="left").fillna(0)
    ringkasan.to_csv(RINGKASAN_PRAPROCESSING, index=False, encoding="utf-8")

    per_inf = (komentar_bersih.groupby("username").size().rename("jumlah_komentar")
               .reset_index().sort_values("jumlah_komentar", ascending=False))
    per_inf.to_csv(KOMENTAR_PER_INFLUENCER, index=False, encoding="utf-8")

    print(f"[Cleaning] Komentar bersih: {len(komentar_bersih):,} dari {len(comments):,} mentah -> {output_csv}")
    return komentar_bersih


komentar_bersih = clean_comments()

## STEP 3 - Sampel gold-standard untuk labeling manual

In [ ]:
def prepare_labeling_sample(output_csv: Path = SAMPEL_LABELING) -> pd.DataFrame:
    if output_csv.exists():
        print(f"[Sampling] Sudah ada -> {output_csv} (skip)")
        return pd.read_csv(output_csv)

    komentar_bersih = clean_comments()
    N = len(komentar_bersih)

    n0 = Z_SCORE ** 2 * P_PROPORTION * (1 - P_PROPORTION) / MARGIN_OF_ERROR ** 2
    n_sample_min = math.ceil(n0 / (1 + (n0 - 1) / N))
    print(f"[Sampling] Populasi komentar bersih: {N:,} | Sampel minimum (MoE 3%): {n_sample_min:,}")

    if N_SAMPLE_PAKAI < n_sample_min:
        raise ValueError(f"N_SAMPLE_PAKAI ({N_SAMPLE_PAKAI}) lebih kecil dari sampel minimum ({n_sample_min})")

    n_ambil = min(N_SAMPLE_PAKAI, N)
    sampel = komentar_bersih.sample(n=n_ambil, random_state=SAMPLE_SEED).reset_index(drop=True)
    sampel["label_manual"] = ""
    sampel["catatan"] = ""

    k = round(len(sampel) * UJI_ULANG_PROP)
    sampel["uji_ulang"] = 0
    sampel.loc[:k - 1, "uji_ulang"] = 1

    sampel.to_csv(output_csv, index=False, encoding="utf-8")

    moe_tercapai = Z_SCORE * math.sqrt(P_PROPORTION * (1 - P_PROPORTION) / n_ambil * (N - n_ambil) / (N - 1))
    print(f"[Sampling] {len(sampel):,} sampel untuk dilabeli | MoE tercapai: {moe_tercapai * 100:.2f}% | "
          f"dilabeli 2x: {k:,} -> {output_csv}")
    return sampel


sampel = prepare_labeling_sample()

## Stats/Grafik -- EDA Preprocessing Komentar

In [ ]:
import matplotlib.pyplot as plt

eda = pd.read_csv(OUTPUT_DIR / "eda_per_influencer.csv")
log = pd.read_csv(OUTPUT_DIR / "log_praprocessing.csv")
ringkasan = pd.read_csv(OUTPUT_DIR / "ringkasan_praprocessing.csv")
per_inf = pd.read_csv(OUTPUT_DIR / "komentar_per_influencer.csv").sort_values("jumlah_komentar", ascending=False)

print(f"Influencer: {len(eda)}")
print(f"Video target: {int(eda.target_video.sum()):,}")
print(f"Video dengan komentar: {int(eda.video_ada_komentar.sum()):,}")
print(f"Video tanpa komentar: {int(eda.video_hilang.sum()):,}")

### 1. Log Pembersihan Tahap-demi-Tahap

In [ ]:
print(log[["langkah", "sebelum", "sesudah", "dibuang"]])

awal = log.iloc[0]["sebelum"]
akhir = log.iloc[-1]["sesudah"]
print(f"\nTotal komentar mentah : {awal:,.0f}")
print(f"Total komentar bersih : {akhir:,.0f}")
print(f"Dibuang keseluruhan   : {awal - akhir:,.0f} ({(awal - akhir) / awal * 100:.1f}%)")

### 2. Komentar Bersih per Influencer

In [ ]:
plot_df = per_inf.iloc[::-1]
fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(plot_df))))
ax.barh(plot_df["username"], plot_df["jumlah_komentar"], color="#4C72B0")
ax.set_xlabel("Jumlah komentar bersih")
ax.set_title("Jumlah Komentar Bersih per Influencer")
for y, v in enumerate(plot_df["jumlah_komentar"]):
    ax.text(v, y, f" {v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

print(per_inf)

### 3. Video Target vs Video yang Benar-Benar Dapat Komentar

In [ ]:
print(ringkasan[["username", "target_video", "video_ada_komentar", "video_hilang",
                    "video_setelah_bersih", "komentar_setelah_bersih"]].sort_values("video_hilang", ascending=False))

print(f"\nTotal video hilang (target tapi 0 komentar): {int(eda.video_hilang.sum())}")